# FedRBE correction for Quartet multiomics

This notebook prepares FedRBE-style client folders and creates federated batch-correction outputs for the full Quartet multiomics dataset. The source batches have 12 samples each, but the FedRBE design has 18 columns, so single batches cannot be clients. Batches are therefore grouped deterministically into 2-batch clients, with the last client holding 3 batches.

In [ ]:
from pathlib import Path
import json
import os
import sys

WORKDIR = Path.cwd() if Path("figshare_data").exists() else Path("evaluation_data/multiomics")
REPO_ROOT = WORKDIR.resolve().parents[1]
if str(WORKDIR.resolve()) not in sys.path:
    sys.path.insert(0, str(WORKDIR.resolve()))

from fedrbe_multiomics_utils import MODALITIES, run_all_fedsim, prepare_fedrbe_clients

print(f"WORKDIR: {WORKDIR.resolve()}")
print(f"Modalities: {MODALITIES}")

## Prepare FedRBE client folders

The client folders follow the same file conventions as the other FedRBE datasets: `intensities_log_UNION.tsv`, `design.tsv`, and `config.yml`. The original Quartet `batch` remains the `batch_col` inside each grouped client.

In [ ]:
client_summaries = []
for modality in MODALITIES:
    summary = prepare_fedrbe_clients(WORKDIR, modality)
    client_summaries.append(summary)
    print(f"\n{modality}")
    print(summary[["client", "position", "batch_codes", "n_samples", "reference_batch"]])

all_clients = __import__("pandas").concat(client_summaries, axis=0, ignore_index=True)
all_clients.to_csv(WORKDIR / "before_fedrbe" / "fedrbe_client_groups.tsv", sep="\t", index=False)
all_clients

## Local FedRBE-equivalent correction

The current Python environment lacks the `FeatureCloud`, `docker`, `PyYAML`, and `bios` packages needed by the FeatureCloud runner. To still create reproducible correction outputs, this cell runs the same XTX/XTY aggregation and batch-effect subtraction used by the app, using the prepared client folders and design coding.

In [ ]:
correction_summary = run_all_fedsim(WORKDIR)
correction_summary

## Output files

Per-modality simulated FedRBE outputs are written beside the central limma outputs. The combined all-modality matrix uses the same equal-weight block construction as the central correction notebook.

In [ ]:
for modality in MODALITIES:
    out_dir = WORKDIR / "after" / modality
    print(modality)
    for name in ["FedSim_corrected_data.tsv", "fedrbe_client_groups.tsv"]:
        path = out_dir / name
        print(f"  {path}: {path.exists()}")
    indiv = out_dir / "individual_results_fedsim"
    print(f"  individual_results_fedsim clients: {len(list(indiv.glob('*')))}")

combined_path = WORKDIR / "after" / "all_modalities_fedsim_kmeans_matrix.tsv"
print(f"Combined FedSim k-means matrix: {combined_path} ({combined_path.exists()})")
print(f"Summary: {WORKDIR / 'after' / 'fedsim_correction_summary.tsv'}")

## Optional FeatureCloud app run

Set `RUN_FEATURECLOUD = True` only in an environment with the FeatureCloud Python package, Docker Python SDK, PyYAML, and a running Docker daemon. The local `featurecloud.ai/bcorrect:latest` image is required.

In [ ]:
RUN_FEATURECLOUD = False

if RUN_FEATURECLOUD:
    import time
    import zipfile
    import pandas as pd
    from copy import deepcopy

    sys.path.insert(0, str(REPO_ROOT))
    from evaluation_utils import featurecloud_api_extension as util

    app_image_name = "featurecloud.ai/bcorrect:latest"
    base_config = {
        "flimmaBatchCorrection": {
            "data_filename": "intensities_log_UNION.tsv",
            "design_filename": "design.tsv",
            "expression_file_flag": True,
            "index_col": "rowname",
            "batch_col": "batch",
            "covariates": ["D6", "F7", "M8"],
            "separator": "\t",
            "design_separator": "\t",
            "normalizationMethod": None,
            "smpc": False,
            "min_samples": 0,
            "position": None,
            "reference_batch": False,
        }
    }

    def postprocess_results(experiment, result_files_zipped, result_filename):
        time.sleep(10)
        result_folder = Path(result_filename).parent
        individual_results_dir = result_folder / "individual_results"
        individual_results_dir.mkdir(parents=True, exist_ok=True)
        client_names = [Path(p).name for p in experiment.clients]

        for zip_path, client_name in zip(result_files_zipped, client_names):
            client_result_dir = individual_results_dir / client_name
            client_result_dir.mkdir(parents=True, exist_ok=True)
            with zipfile.ZipFile(zip_path, "r") as zf:
                for fname in ["only_batch_corrected_data.csv", "full_corrected_data.csv", "report.txt"]:
                    if fname in zf.namelist():
                        zf.extract(fname, client_result_dir)

        final_df = None
        for client_name in client_names:
            result_file = individual_results_dir / client_name / "only_batch_corrected_data.csv"
            client_df = pd.read_csv(result_file, sep="\t", index_col=0)
            final_df = client_df if final_df is None else pd.concat([final_df, client_df], axis=1)
        final_df.to_csv(result_filename, sep="\t")
        return final_df

    for modality in MODALITIES:
        groups = __import__("pandas").read_csv(WORKDIR / "before_fedrbe" / modality / "fedrbe_client_groups.tsv", sep="\t")
        clients = [str(WORKDIR / "before_fedrbe" / modality / client) for client in groups["client"]]
        config_changes = []
        for _, row in groups.iterrows():
            reference = row["reference_batch"] if isinstance(row["reference_batch"], str) and row["reference_batch"] != "False" else False
            config_changes.append({
                "flimmaBatchCorrection": {
                    "position": int(row["position"]),
                    "reference_batch": reference,
                    "smpc": False,
                }
            })
        experiment = util.Experiment(
            name=f"Multiomics {modality}",
            fc_data_dir=str(WORKDIR),
            clients=clients,
            app_image_name=app_image_name,
            config_files=[deepcopy(base_config) for _ in clients],
            config_file_changes=config_changes,
        )
        result_files_zipped, _, _ = experiment.run_test()
        postprocess_results(experiment, result_files_zipped, WORKDIR / "after" / modality / "FedApp_corrected_data.tsv")
else:
    print("FeatureCloud run skipped. Local FedSim outputs above are ready for downstream checks.")